# 01 — Data: Getting a Real Pretraining Corpus from Hugging Face

**Goal:** build a real, filtered, deduplicated, tokenized pretraining corpus
that you will actually train on in notebook 04 — and understand every filter
you applied and why.

**Time:** 30–60 min. **Disk:** ~2 GB for the default 500M-token slice.

## Why this notebook comes first

The uncomfortable truth of the last few years: **at a fixed compute budget,
data quality beats architecture, and it isn't close.** FineWeb-Edu showed that
filtering CommonCrawl by an "educational value" classifier let a model match
the benchmark scores of a model trained on ~5–10× more unfiltered tokens.
SmolLM2 (135M/360M/1.7B) is competitive with much larger contemporaries almost
entirely because of its data mix.

Nobody writes the blog post "we improved our data pipeline by 4%." Everybody
writes the one about the new attention variant. Do not be fooled by what gets
written about.

## The corpora that matter (as of mid-2026)

| Dataset | Size | What it is | Use it for |
|---|---|---|---|
| `HuggingFaceFW/fineweb-edu` | 1.3T tok | CommonCrawl filtered by an edu-quality classifier | **Default choice for pretraining** |
| `HuggingFaceFW/fineweb` | 15T tok | Same pipeline, unfiltered by quality | When you need raw scale |
| `HuggingFaceTB/smollm-corpus` | 600B tok | fineweb-edu-dedup + Cosmopedia v2 + Python-Edu | Great small-model recipe |
| `bigcode/the-stack-v2` | 900B+ tok | Permissively-licensed source code | Code ability |
| `open-web-math/open-web-math` | 15B tok | Math-heavy web text | Math/reasoning ability |
| `HuggingFaceTB/cosmopedia-v2` | 28B tok | *Synthetic* textbooks generated by an LLM | Density; mix, don't use alone |
| `roneneldan/TinyStories` | ~500M tok | Simple synthetic children's stories | **Debugging / first run** |

We'll use **TinyStories** for the smoke test (a 10M model can learn it in
minutes, which makes bugs obvious) and **fineweb-edu** for the real run.

In [ ]:
import os
from pathlib import Path

import numpy as np

DATA = Path("../data")
DATA.mkdir(parents=True, exist_ok=True)

# Be polite to the Hub and fast on repeat runs.
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
print("data dir:", DATA.resolve())

## Streaming vs downloading — the thing that saves your disk

`fineweb-edu` is ~10 TB. You cannot download it, and you don't need to.
`streaming=True` gives you an `IterableDataset` that pulls shards over HTTP
as you iterate.

The trade-offs are real, so know them:

| | `streaming=False` | `streaming=True` |
|---|---|---|
| disk | full dataset | ~nothing |
| `len()` | works | **fails** — length is unknown |
| random access `ds[42]` | works | **fails** — forward-only |
| shuffling | exact | approximate (buffer) |
| resumable mid-epoch | easy | awkward |
| needs network during training | no | **yes** |

The standard practice, which we follow: **stream once, tokenize, write to a
flat binary file.** After that, training reads from a local `.bin` via
`np.memmap` — no network, no HF library in the hot loop, no `__getitem__`
overhead. This is exactly what nanoGPT and nanochat do.

In [ ]:
from datasets import load_dataset

# A named config selects a CommonCrawl snapshot. "sample-10BT" is a
# pre-sampled 10-billion-token slice — plenty for us, and much faster to
# stream than the full set.
FINEWEB_CONFIG = "sample-10BT"

stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name=FINEWEB_CONFIG,
    split="train",
    streaming=True,
)

print("features:", stream.features)
print("\n--- first 2 documents ---")
for i, doc in enumerate(stream.take(2)):
    print(f"\n[doc {i}]")
    for k, v in doc.items():
        if k == "text":
            print(f"  text ({len(v)} chars): {v[:280]!r} ...")
        else:
            print(f"  {k}: {v}")

Note the `score` field — that's the educational-quality classifier's rating
(0–5). The dataset is already filtered to score ≥ 3. Raising the threshold
further trades quantity for quality; at our tiny scale, quality tends to win.

## Quality filtering: the heuristics everyone actually uses

Even "clean" corpora contain junk that wastes your compute. The filters below
are the standard ones from the C4, Gopher, and RefinedWeb papers. They are
unglamorous and they matter.

Each one exists because of a specific failure:

- **too short** → no learnable long-range structure
- **low mean word length / very high** → gibberish, or base64 blobs
- **few lines end in punctuation** → navigation menus and link farms
- **high symbol-to-word ratio** → code fragments, math dumps, tables of junk
- **high duplicate-line fraction** → boilerplate headers/footers repeated
- **low alphabetic fraction** → data tables masquerading as prose

In [ ]:
import re
from collections import Counter

WORD_RE = re.compile(r"\S+")


def quality_stats(text: str) -> dict:
    """Compute the standard Gopher-style quality signals for one document."""
    words = WORD_RE.findall(text)
    n_words = len(words)
    if n_words == 0:
        return {"n_words": 0}

    lines = [ln for ln in text.split("\n") if ln.strip()]
    n_lines = max(len(lines), 1)

    mean_word_len = sum(len(w) for w in words) / n_words
    ends_punct = sum(1 for ln in lines if ln.rstrip().endswith((".", "!", "?", '"', "'"))) / n_lines
    symbol_ratio = sum(text.count(c) for c in "#…") / n_words
    ellipsis_lines = sum(1 for ln in lines if ln.rstrip().endswith("…")) / n_lines
    alpha_frac = sum(1 for w in words if any(c.isalpha() for c in w)) / n_words
    dup_lines = 1.0 - (len(set(lines)) / n_lines)

    return {
        "n_words": n_words,
        "mean_word_len": mean_word_len,
        "frac_lines_end_punct": ends_punct,
        "symbol_to_word": symbol_ratio,
        "frac_lines_ellipsis": ellipsis_lines,
        "frac_words_alpha": alpha_frac,
        "frac_dup_lines": dup_lines,
    }


def passes_quality(text: str, *, min_words: int = 50) -> tuple[bool, str]:
    """Return (keep?, reason_if_rejected). Thresholds from Gopher / RefinedWeb."""
    s = quality_stats(text)
    if s["n_words"] < min_words:
        return False, "too_short"
    if not (3.0 <= s["mean_word_len"] <= 10.0):
        return False, "word_len"
    if s["frac_lines_end_punct"] < 0.15:
        return False, "no_punctuation"     # menus, link lists
    if s["symbol_to_word"] > 0.10:
        return False, "symbol_heavy"
    if s["frac_lines_ellipsis"] > 0.30:
        return False, "ellipsis_spam"      # truncated teaser text
    if s["frac_words_alpha"] < 0.70:
        return False, "not_prose"
    if s["frac_dup_lines"] > 0.30:
        return False, "dup_lines"
    return True, "ok"


# Sanity-check the filter on documents we construct to be obviously good or bad.
probes = {
    "good prose": (
        "The mitochondrion is an organelle found in most eukaryotic cells. "
        "It generates most of the cell's supply of adenosine triphosphate. "
        "Mitochondria have their own genome, a fact which supports the "
        "endosymbiotic theory of their origin. This theory proposes that they "
        "descend from free-living bacteria engulfed by an ancestral cell. "
    ) * 3,
    "nav menu": "\n".join(["Home", "About", "Contact", "Products", "Login"] * 12),
    "boilerplate": "\n".join(["Copyright 2024 All Rights Reserved."] * 30),
    "teaser spam": "\n".join([f"Read more about topic {i}…" for i in range(30)]),
    "too short": "Hello world.",
}
for name, txt in probes.items():
    keep, why = passes_quality(txt)
    print(f"{name:<14} keep={str(keep):<5} reason={why}")

Now measure the filter against the real stream, and see what it rejects.
**Always look at what your filter throws away.** A filter that rejects 90% of
your corpus is a bug, not a feature.

In [ ]:
SAMPLE_N = 2000

reasons = Counter()
kept_docs = []
for doc in load_dataset(
    "HuggingFaceFW/fineweb-edu", name=FINEWEB_CONFIG, split="train", streaming=True
).take(SAMPLE_N):
    keep, why = passes_quality(doc["text"])
    reasons[why] += 1
    if keep and len(kept_docs) < 3:
        kept_docs.append(doc["text"])

total = sum(reasons.values())
print(f"of {total} fineweb-edu documents:\n")
for why, n in reasons.most_common():
    print(f"  {why:<18} {n:>5}  ({100*n/total:5.1f}%)")
print(f"\nkeep rate: {100*reasons['ok']/total:.1f}%")

A keep rate in the 80–95% range is what you want on an already-filtered corpus
like fineweb-edu — you're catching residual junk, not re-doing their work. On
raw CommonCrawl the same filters would drop far more.

## Deduplication: why it's not optional

Duplicated training data causes two concrete problems:

1. **Memorization.** Sequences seen many times get memorized verbatim rather
   than generalized from. This is also how training data gets regurgitated.
2. **Wasted compute.** A duplicate is a gradient step you already took.

Exact dedup (hash the document) is easy but only catches identical text. Near
duplicates — the same article on 50 sites with different boilerplate — need
**MinHash + LSH**, which estimates Jaccard similarity between shingle sets
without comparing every pair.

We implement it here because the idea is genuinely simple once you see it.

In [ ]:
import hashlib


def shingles(text: str, k: int = 5) -> set[str]:
    """Set of k-word sequences. Two docs are 'similar' if their shingle sets are."""
    words = text.lower().split()
    if len(words) < k:
        return {" ".join(words)} if words else set()
    return {" ".join(words[i : i + k]) for i in range(len(words) - k + 1)}


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def minhash_signature(sh: set[str], n_perm: int = 128) -> np.ndarray:
    """MinHash: for each of n_perm hash functions, keep the minimum hash value.

    The key property: P(min_hash_i(A) == min_hash_i(B)) == Jaccard(A, B).
    So comparing two 128-int signatures estimates Jaccard without touching the
    original sets. That turns an O(n^2) set-comparison problem into something
    you can index.
    """
    if not sh:
        return np.full(n_perm, np.iinfo(np.uint64).max, dtype=np.uint64)
    sig = np.full(n_perm, np.iinfo(np.uint64).max, dtype=np.uint64)
    for s in sh:
        base = int.from_bytes(hashlib.sha1(s.encode()).digest()[:8], "big")
        # Cheap family of hash functions: (a*h + b) mod prime, for varying a,b.
        for i in range(n_perm):
            h = (base * (2 * i + 1) + i * 0x9E3779B9) % (2**61 - 1)
            if h < sig[i]:
                sig[i] = h
    return sig


def estimated_jaccard(sig_a: np.ndarray, sig_b: np.ndarray) -> float:
    return float((sig_a == sig_b).mean())


# Demonstrate: near-duplicates should score high, unrelated text low.
doc_a = "The quick brown fox jumps over the lazy dog. " * 20
doc_b = "The quick brown fox jumps over the lazy dog. " * 19 + "An extra sentence here. "
doc_c = "Completely unrelated content about marine biology and coral reefs. " * 20

sa, sb, sc = (minhash_signature(shingles(d)) for d in (doc_a, doc_b, doc_c))
print(f"A vs B  true={jaccard(shingles(doc_a), shingles(doc_b)):.3f}  "
      f"minhash={estimated_jaccard(sa, sb):.3f}   <- near-duplicate")
print(f"A vs C  true={jaccard(shingles(doc_a), shingles(doc_c)):.3f}  "
      f"minhash={estimated_jaccard(sa, sc):.3f}   <- unrelated")

**In production, use a library.** Our loop is O(shingles × permutations) in
pure Python — fine for understanding, far too slow for a real corpus. Use
`datasketch` (`MinHashLSH`) or, at scale, the `datatrove` library that the
FineWeb team wrote for exactly this. Below we use plain exact-hash dedup,
which is cheap and catches the common case.

## First: what is a token?

This word is about to appear in every cell for the rest of the course, so it
is worth ten lines now.

A neural network cannot read text. It multiplies matrices of numbers. So
before any model sees your corpus, the text has to become a list of integers.
A **token** is one of those integers — and, equivalently, the chunk of text it
stands for.

The obvious choices are both bad:

- **One token per character.** Vocabulary of ~100, but sequences become
  enormous. "understanding" is 13 steps of computation instead of 1–2, and
  attention cost grows with the *square* of sequence length.
- **One token per word.** Short sequences, but the vocabulary is unbounded —
  every typo, name, and inflection is a new word — and anything unseen at
  training time becomes `<UNK>`, which destroys information.

**Byte-Pair Encoding (BPE)** splits the difference. Start from bytes, then
repeatedly merge the most frequent adjacent pair into a new token. Common
words end up as a single token; rare words survive as a handful of pieces;
nothing is ever unrepresentable, because you can always fall back to bytes.

So a token is usually *a common word or a word-fragment*, and the useful rule
of thumb for English is:

> **1 token ≈ 4 characters ≈ 0.75 words.**
> 1,000 tokens is roughly 750 words, or about a page and a half.

Two consequences that will bite you later if you don't internalize them now:

- Token counts, not word counts, are what fills a context window, what you pay
  for on an API, and what `block_size` measures.
- The **leading space is part of the token**. `" the"` and `"the"` are
  *different* ids. This is why a prompt should never end with a trailing space
  — you hand the model a token it almost never saw during training. Notebook
  02 builds a tokenizer from scratch and makes this concrete.

The cell below loads GPT-2's tokenizer and shows all of that on real text.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
EOT = tok.eos_token_id  # 50256, the document separator
print(f"vocab size {tok.vocab_size}, eot id {EOT}")

# See it happen. Note the Ġ — that is how GPT-2 draws a leading space.
demo = "The cat sat on the mat. Antidisestablishmentarianism!"
ids = tok.encode(demo)
print(f"\ntext      : {demo!r}")
print(f"ids       : {ids}")
print(f"pieces    : {tok.convert_ids_to_tokens(ids)}")
print(f"\n{len(demo)} chars -> {len(ids)} tokens "
      f"({len(demo)/len(ids):.2f} chars/token)")

# Common words are one token; rare long words shatter into pieces.
for w in ["the", " the", "cat", "Antidisestablishmentarianism", "1234567"]:
    print(f"  {w!r:<32} -> {len(tok.encode(w))} token(s) {tok.encode(w)}")

Read that output carefully — three things are visible in it.

**`" the"` and `"the"` have different ids.** Same letters, different token,
because the space is baked in.

**`Antidisestablishmentarianism` costs 5–8 tokens** while `the` costs 1. The
tokenizer spends its vocabulary budget on what is *frequent*, not on what is
long. This is also why tokenizers are bad at arithmetic: `1234567` splits into
arbitrary chunks that carry no numeric meaning, which is a real cause of
digit-level mistakes in LLMs. Llama 3 changed its tokenizer to split every
digit separately for exactly this reason.

**The chars/token ratio lands near 4.** That is the compression number
notebook 02 measures properly, and it directly sets how much text fits in your
context window.

### Practice — guess before you run

Reading about tokens does not build intuition; predicting does. For each string
below, **write down your guess** for how many tokens it becomes, then run the
cell.

Use the rule of thumb — 1 token ≈ 4 characters — then adjust for what you just
learned: common words are cheap, rare words and numbers are expensive.

1. `"cat"`
2. `"The quick brown fox jumps over the lazy dog."`
3. `"3.14159265358979"`
4. `"antidisestablishmentarianism"`
5. `"🎉🎉🎉"`
6. `"    "` (four spaces)

Most people get 1 and 2 close, and are badly wrong on 3, 5 and 6. Those three
are where the real intuition lives.

In [ ]:
practice = [
    "cat",
    "The quick brown fox jumps over the lazy dog.",
    "3.14159265358979",
    "antidisestablishmentarianism",
    "🎉🎉🎉",
    "    ",
]

print(f"{'text':<48}{'chars':>6}{'tokens':>8}{'ch/tok':>8}")
print("-" * 70)
for s in practice:
    n = len(tok.encode(s))
    print(f"{s!r:<48}{len(s):>6}{n:>8}{len(s)/n:>8.2f}")

print("\nWhere the surprises come from:")
print("  numbers  -> split into arbitrary chunks, no numeric meaning")
print("  emoji    -> several tokens each; they are multi-byte characters")
print("  spaces   -> runs of whitespace are their own tokens (matters for code)")

The whitespace result is why **tokenizers matter for code**. Python is
indentation-heavy, and a tokenizer that spends one token per space burns your
context window on empty air. Code-focused tokenizers add tokens for runs of 4,
8, 12 spaces specifically to fix this — and it is a large part of why
code-specific models feel cheaper to run on code.

The emoji result is why **non-English text costs more**. Notebook 02 measures
this properly and calls it the multilingual tax: the same sentence in Thai or
Hindi can cost 3–5× the tokens of its English translation, meaning less context
and higher API bills for the same content.

## Building the corpus: stream → filter → dedup → tokenize → `.bin`

Now we turn a whole dataset into tokens. The output format is deliberately
dumb: **one flat array of `uint16` token ids**, documents separated by an
end-of-text token. That's it. No JSON, no per-example padding, no
`__getitem__`. Training samples a random offset and slices `block_size + 1`
tokens.

It helps to picture the file as one unbroken ribbon of numbers:

```
[   464  3797  3332 ... 13 50256   818  1110 ... 50256  ... ]
 \________ document 1 _______/ ^     \____ document 2 ____/ ^
                               |                            |
                         EOT (50256)                   EOT (50256)
```

**Why one flat array instead of a list of documents?** Because training wants
fixed-length windows, and a flat array gives them for free: pick a random
offset `i`, slice `tokens[i : i + block_size + 1]`, done. No padding, no
bucketing, no wasted compute on `<PAD>`. Windows occasionally straddle the EOT
boundary and contain the tail of one document plus the head of the next — that
is fine and even useful, since it teaches the model that EOT means "the topic
is about to change completely."

**Why `uint16`?** Two bytes per token, and GPT-2's vocabulary is
50257 < 65536, so every id fits. That keeps 500M tokens at 1 GB instead of 2 GB
under `uint32`. If you ever use a vocabulary larger than 65536 — Llama 3's is
128k — this silently wraps around and corrupts your data, so you must switch to
`uint32` and pay double the disk.

We use the GPT-2 tokenizer here so this notebook stands alone; notebook 02
builds a tokenizer from scratch and you can rerun this with your own.

In [ ]:
def build_bin(
    out_path: Path,
    hf_name: str,
    *,
    hf_config: str | None = None,
    target_tokens: int = 50_000_000,
    text_key: str = "text",
    apply_filter: bool = True,
    dedup: bool = True,
    report_every: int = 25_000,
) -> dict:
    """Stream a HF dataset to a flat uint16 token file. Returns statistics."""
    out_path.parent.mkdir(parents=True, exist_ok=True)

    ds = load_dataset(hf_name, name=hf_config, split="train", streaming=True)
    seen_hashes: set[bytes] = set()
    stats = Counter()
    n_tokens = 0

    buf: list[int] = []
    FLUSH_AT = 1_000_000  # tokens held in RAM before writing

    with open(out_path, "wb") as f:
        for doc in ds:
            text = doc.get(text_key)
            if not text:
                stats["empty"] += 1
                continue

            if apply_filter:
                keep, why = passes_quality(text)
                if not keep:
                    stats[f"drop_{why}"] += 1
                    continue

            if dedup:
                h = hashlib.sha1(text.encode("utf-8", "ignore")).digest()
                if h in seen_hashes:
                    stats["drop_exact_dup"] += 1
                    continue
                seen_hashes.add(h)

            ids = tok.encode(text)
            ids.append(EOT)
            buf.extend(ids)
            n_tokens += len(ids)
            stats["kept"] += 1

            if len(buf) >= FLUSH_AT:
                np.array(buf, dtype=np.uint16).tofile(f)
                buf.clear()

            if n_tokens >= target_tokens:
                break
            if stats["kept"] % report_every == 0 and stats["kept"]:
                print(f"  {stats['kept']:>7} docs | {n_tokens/1e6:6.1f}M tokens")

        if buf:
            np.array(buf, dtype=np.uint16).tofile(f)

    stats["total_tokens"] = n_tokens
    return dict(stats)

### Build 1 — TinyStories (the smoke-test corpus)

**In plain language: what is about to happen, and why.**

The next cell downloads about 92,000 short children's stories, converts them
into numbers, and saves those numbers to a single file on your disk. It takes
under a minute. When it finishes you will have a 40 MB file called
`tinystories_train.bin`.

That file is **not a model**. Nothing has been trained yet. Think of it as the
textbook — you are printing the book now, and in notebook 04 the model will sit
down and read it. Building the textbook and reading it are separate jobs, which
is why they are separate notebooks.

**What is TinyStories?** A dataset someone built by asking GPT-4 to write
millions of very short stories using only words a three- or four-year-old
knows. They look like this:

> *Once upon a time there was a little girl named Lily. She liked to play with
> her red ball. One day the ball rolled under the bed and Lily was sad.*

Simple vocabulary, simple grammar, always a beginning and an end.

**Why start with children's stories instead of real text?** Because a small
model can actually learn them. Real web text needs a model with billions of
parameters to do anything useful. TinyStories is simple enough that a model
1/1000th that size — one you can train on your own GPU in fifteen minutes —
genuinely learns to write coherent English.

It is the empty car park before you drive on the motorway. You are not trying
to build something impressive here. You are checking that your car steers.

**So what should you expect at the end of all this?** After notebook 04 trains
on this file, your model will produce things like:

> *Once upon a time there was a little boy named Tim. Tim had a big red dog.
> The dog liked to run in the park. Tim was happy.*

**That is success.** Not ChatGPT — not close. It cannot answer questions, hold
a conversation, do arithmetic, or tell you anything true about the world. It
writes simple stories, because simple stories are all it has ever seen. What
you will have proven is that *the machinery works*: your data pipeline, your
tokenizer, your model architecture, and your training loop are all correct end
to end. That is genuinely the hard part, and everything after it is scale.

**The honest version of why this matters.** If you skipped straight to real web
text and your model produced gibberish, you would have no idea whether you had
a bug or simply needed more compute. Debugging that is miserable. Here, if the
model cannot learn TinyStories, you have a **bug** — full stop, no ambiguity,
because everyone else's models learn TinyStories easily. That certainty is what
you are buying.

**Where this sits in the course:**

```
  you are here
       |
       v
  01 build the textbook  ->  04 model reads it  ->  07-13 teach it manners
  (numbers on disk)          (a model that          (answer questions,
                              writes stories)        follow instructions)
```

Now the technical framing of the same thing. TinyStories is simple enough that
a 10M-parameter model learns fluent English on it in ~15 minutes. That makes it
perfect for verifying your training loop is correct: if your model *can't*
learn TinyStories, you have a bug, not a scale problem. Debug on this, then
scale up.

Note we skip the prose filter — TinyStories are short by design and the
`min_words` rule would reject most of them.

**What you should see.** Progress lines every 25,000 documents, then a stats
dictionary. Expect roughly:

```
  25000 docs |    5.6M tokens
  50000 docs |   11.1M tokens
  75000 docs |   16.6M tokens

 {'kept': 91158, 'empty': 12, 'drop_exact_dup': 950, 'total_tokens': 20000043}
```

Takes 20–40 s and writes a **40 MB** file. Your numbers will differ by a few
percent — streaming order is not guaranteed stable — but the *shape* should
match. Here is how to read every field:

| Field | Meaning | Healthy value here |
|---|---|---|
| `kept` | documents tokenized and written | ~91k |
| `empty` | documents with no `text` field | a handful |
| `drop_exact_dup` | byte-identical repeats caught by the SHA-1 set | ~1% |
| `total_tokens` | tokens written, the number that matters | ≥ target |

Three sanity checks worth doing every single time you build a corpus:

**1. `total_tokens` slightly exceeds your target.** We requested 20,000,000 and
got 20,000,043. The loop only checks the budget *between* documents, so it
overshoots by at most one document. Wildly over means your break condition is
broken; wildly under means the stream ran dry.

**2. Tokens per document ≈ 220.** `20.0M / 91k ≈ 220 tokens`, about 165 words —
exactly right for a children's story. If this came out at 5 you are tokenizing
empty strings; if it came out at 5,000 you are reading the wrong field and
probably concatenated the whole split into one row.

**3. File size ≈ 2 bytes × tokens.** 20M tokens → ~40 MB, because `uint16`.
If the file is 80 MB, something upcast to `uint32` and you are wasting half
your disk. If it is 40 KB, the buffer never flushed.

The `drop_exact_dup` count deserves a second look: ~950 out of ~92k documents
are **byte-identical duplicates** in a *curated synthetic* dataset. Real web
scrapes are far worse. That is the argument for the dedup section above, made
with a number instead of an assertion.

In [ ]:
tiny_path = DATA / "tinystories_train.bin"
if tiny_path.exists():
    print(f"exists: {tiny_path} ({tiny_path.stat().st_size/1e6:.1f} MB) — delete to rebuild")
else:
    s = build_bin(
        tiny_path,
        "roneneldan/TinyStories",
        target_tokens=20_000_000,
        apply_filter=False,   # stories are intentionally short & simple
        dedup=True,
    )
    print("\n", s)

Now prove the file is what we think it is, rather than trusting the stats.

In [ ]:
arr = np.memmap(tiny_path, dtype=np.uint16, mode="r")
print(f"tokens on disk : {len(arr):,}")
print(f"file size      : {tiny_path.stat().st_size/1e6:.1f} MB "
      f"({tiny_path.stat().st_size / len(arr):.0f} bytes/token)")
print(f"id range       : {arr.min()}–{arr.max()}  (must be < {tok.vocab_size})")
print(f"docs (EOT count): {int((arr == EOT).sum()):,}")
print(f"mean tokens/doc : {len(arr) / max(1, int((arr == EOT).sum())):.0f}")

print("\n--- first 300 tokens decoded ---")
print(tok.decode(arr[:300].tolist()))

### Build 2 — FineWeb-Edu (the real corpus)

Set `TARGET` by what you want to do:

| tokens | disk | stream time | trains a... |
|---|---|---|---|
| 50M | 100 MB | ~3 min | toy model, quick iteration |
| 500M | 1 GB | ~20 min | **good 124M model (this course's default)** |
| 2.5B | 5 GB | ~2 h | Chinchilla-optimal 124M model |

Chinchilla says ~20 tokens per parameter is compute-optimal, so 124M × 20 ≈
2.5B tokens. But "compute-optimal" means *best loss for a fixed training
budget* — if you plan to run inference a lot, training a smaller model on more
tokens is the better deal, which is why Llama and SmolLM train far past
Chinchilla. We'll discuss this properly in notebook 06.

In [ ]:
TARGET = 500_000_000

fw_path = DATA / "fineweb_edu_train.bin"
if fw_path.exists():
    print(f"exists: {fw_path} ({fw_path.stat().st_size/1e9:.2f} GB) — delete to rebuild")
else:
    print(f"building {TARGET/1e6:.0f}M tokens — go make coffee")
    s = build_bin(
        fw_path,
        "HuggingFaceFW/fineweb-edu",
        hf_config=FINEWEB_CONFIG,
        target_tokens=TARGET,
        apply_filter=True,
        dedup=True,
    )
    print("\nfinal stats:")
    for k, v in sorted(s.items()):
        print(f"  {k:<24} {v}")

## Hold out a validation split

**Do this before you train, not after.** Validation loss on data the model has
never seen is your only honest signal for overfitting. Taking the tail of the
file is fine here because documents were streamed in a fixed order and are not
sorted by anything meaningful.

In [ ]:
def split_bin(src: Path, val_fraction: float = 0.005) -> tuple[Path, Path]:
    arr = np.memmap(src, dtype=np.uint16, mode="r")
    n_val = int(len(arr) * val_fraction)
    n_train = len(arr) - n_val

    train_p = src.with_name(src.stem + "_split_train.bin")
    val_p = src.with_name(src.stem + "_split_val.bin")

    np.array(arr[:n_train]).tofile(train_p)
    np.array(arr[n_train:]).tofile(val_p)
    print(f"{src.name}: {n_train/1e6:.1f}M train / {n_val/1e6:.2f}M val tokens")
    return train_p, val_p


for p in (tiny_path, fw_path):
    if p.exists() and not p.with_name(p.stem + "_split_val.bin").exists():
        split_bin(p)

## Verify: decode random slices and actually read them

The most common data-pipeline bug is a silent one — off-by-one in the token
stream, wrong dtype, truncated write — and it shows up as a model that trains
but never gets good. **Always eyeball your tokens before training on them.**

In [ ]:
def peek(bin_path: Path, n: int = 3, span: int = 220) -> None:
    arr = np.memmap(bin_path, dtype=np.uint16, mode="r")
    print(f"\n=== {bin_path.name} — {len(arr):,} tokens ({len(arr)*2/1e9:.2f} GB) ===")
    rng = np.random.default_rng(0)
    for i in range(n):
        start = int(rng.integers(0, max(len(arr) - span, 1)))
        chunk = arr[start : start + span].tolist()
        print(f"\n[offset {start:,}] {tok.decode(chunk)!r}")


for p in (tiny_path, fw_path):
    if p.exists():
        peek(p)

Read those samples. They should be coherent natural text. If you see
repeated boilerplate, mojibake, or HTML, fix the filter now — every hour of
training on bad data is an hour wasted.

In [ ]:
# Token-frequency sanity check: does this look like English?
arr = np.memmap(fw_path if fw_path.exists() else tiny_path, dtype=np.uint16, mode="r")
sample = np.array(arr[:2_000_000])
counts = np.bincount(sample, minlength=tok.vocab_size + 1)
top = np.argsort(counts)[::-1][:20]

print(f"{'rank':<6}{'token':<14}{'count':>10}{'%':>8}")
for r, tid in enumerate(top, 1):
    print(f"{r:<6}{tok.decode([int(tid)])!r:<14}{counts[tid]:>10,}{100*counts[tid]/len(sample):>7.2f}%")

nonzero = int((counts > 0).sum())
print(f"\n{nonzero:,} of {tok.vocab_size:,} vocab entries used ({100*nonzero/tok.vocab_size:.1f}%)")

Expect `' the'`, `','`, `' of'`, `'.'` at the top — Zipf's law in action.
If your top token is something weird (a padding token, or `!`), you have a
tokenization bug.

## What we skipped, and when you'd need it

Honest list of what a production pipeline has that this doesn't:

- **PII redaction** — email/phone/address scrubbing. Legally important for
  anything you ship.
- **Language ID filtering** — fastText or CLD3 to keep only your target
  language. FineWeb-Edu is already English-only.
- **Toxicity / NSFW filtering** — classifier-based.
- **Benchmark decontamination** — removing documents containing your eval sets.
  Skipping this is the #1 cause of fake benchmark scores. We cover detection
  in notebook 14.
- **Distributed MinHash-LSH** — real near-dedup at TB scale (`datatrove`).
- **Data mixing / curriculum** — sampling web:code:math at tuned ratios, and
  annealing toward high-quality data at the end of training.

## Checkpoint

- [ ] `data/tinystories_train_split_train.bin` and `_val.bin` exist
- [ ] `data/fineweb_edu_train_split_train.bin` and `_val.bin` exist
- [ ] Decoded samples read like real text
- [ ] You can explain why we stream but train from a local `.bin`

**Next:** `02_tokenizer_from_scratch.ipynb` — we used GPT-2's tokenizer as a
black box. Now build one.